In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

## import

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [4]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention")
cwd = os.getcwd()
print(cwd)

sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention


In [5]:
import scanpy as sc
import spatialdata as sd
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)


In [6]:
import HadmardAttention as HA

/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [7]:
import importlib
import HadmardAttention.tools
import HadmardAttention.model

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

In [ ]:
adata = sdata.tables["table"]
adata

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

In [ ]:
adata_omiCLIP = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/cells.h5ad")
adata_omiCLIP

In [ ]:
# 1. Ensure cell IDs are the index (not just a column)
if 'cell_id' in adata.obs.columns:
    adata.obs.set_index('cell_id', inplace=True)
if 'cell_id' in adata_omiCLIP.obs.columns:
    adata_omiCLIP.obs.set_index('cell_id', inplace=True)

# 2. Align the two objects by cell_id (intersection)
common_ids = adata.obs_names.intersection(adata_omiCLIP.obs_names)

# Optional: check how many matched
print(f"Matched {len(common_ids)} cells out of {adata.n_obs}")

# 3. Reorder both to the same order
adata_c = adata[common_ids, :].copy()
adata_omiCLIP_c = adata_omiCLIP[common_ids, :].copy()

# 4. Add the X_custom matrix to adata_main.obsm
adata_c.obsm['Morpho_Embedding'] = adata_omiCLIP_c.obsm['X_custom']

# 5. Done! Verify
print(adata_c.obsm.keys())

In [ ]:
adata_c.write("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

## Train

In [8]:
adata = sc.read_h5ad("../../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

Founsation Models

In [9]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [10]:
edata.obs_names = [cid[61:] for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['UNI'][edata.obs_names.get_indexer(common_cells)]

In [11]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

In [12]:
from sklearn.decomposition import PCA

pca = PCA(n_components=500)
X_reduced = pca.fit_transform(adata.obsm['Morpho_Embedding'])
adata.obsm['p_Morpho_Embedding'] = X_reduced

If saved before:

In [14]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/UNI_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/hoptimus_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/virchow_adata.h5ad")

Noise

In [ ]:
rng = np.random.default_rng(42)
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal((63173, 500))

In [ ]:
adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Preparing Dataset

In [15]:
adata = HA.prep_adatas(adata, norm=True, log1p=True)
dataset = HA.make_dataset(adata, sparse_graph=True)

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.


In [ ]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)

Expression torch.Size([63173, 2000])
Morpho_Embedding torch.Size([63173, 500])
Neighborhood_Graph torch.Size([2, 505384])


In [ ]:
importlib.reload(HA.dataset)
importlib.reload(HA.model)
importlib.reload(HA)

<module 'HadmardAttention' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention/../../HadmardAttention/__init__.py'>

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

# Train

## Model Type 0

In [16]:
model_type = 0

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


### NOISE

In [ ]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.10502
Epoch 101: loss =  0.08729
Epoch 201: loss =  0.06816
Epoch 301: loss =  0.06208
Epoch 401: loss =  0.05995
Epoch 501: loss =  0.05937
Epoch 601: loss =  0.05865
Epoch 701: loss =  0.05826
Epoch 801: loss =  0.05797
Epoch 901: loss =  0.05714
Epoch 1001: loss =  0.05654
Epoch 1101: loss =  0.05614
Epoch 1201: loss =  0.05599
Epoch 1301: loss =  0.05606
Epoch 1401: loss =  0.05580
Epoch 1501: loss =  0.05569
Epoch 1601: loss =  0.05559
Epoch 1701: loss =  0.05553
Epoch 1801: loss =  0.05548
Epoch 1901: loss =  0.05541
Epoch 2001: loss =  0.05536
Epoch 2101: loss =  0.05531
Epoch 2201: loss =  0.05526
Epoch 2301: loss =  0.05523
Epoch 2401: loss =  0.05520
Epoch 2501: loss =  0.05517
Epoch 2601: loss =  0.05515
Epoch 2701: loss =  0.05515
Epoch 2801: loss =  0.05512
Epoch 2901: loss =  0.05509
Epoch 3001: loss =  0.05509
Epoch 3101: loss =  0.05508
Epoch 3201: loss =  0.05508
Epoch 3301: loss =  0.05499
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_NOISE_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_NOISE_{model_type}.pth',weights_only=True))

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_5k_32_NOISE_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_5k_32_NOISE_{model_type}.pth',weights_only=True))

### h_Optimus

In [ ]:
#H_OPTIMUS
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.10496
Epoch 101: loss =  0.08791
Epoch 201: loss =  0.06765
Epoch 301: loss =  0.06094
Epoch 401: loss =  0.05829
Epoch 501: loss =  0.05719
Epoch 601: loss =  0.05618
Epoch 701: loss =  0.05560
Epoch 801: loss =  0.05530
Epoch 901: loss =  0.05505
Epoch 1001: loss =  0.05478
Epoch 1101: loss =  0.05464
Epoch 1201: loss =  0.05441
Epoch 1301: loss =  0.05426
Epoch 1401: loss =  0.05415
Epoch 1501: loss =  0.05418
Epoch 1601: loss =  0.05394
Epoch 1701: loss =  0.05383
Epoch 1801: loss =  0.05376
Epoch 1901: loss =  0.05370
Epoch 2001: loss =  0.05369
Epoch 2101: loss =  0.05367
Epoch 2201: loss =  0.05361
Epoch 2301: loss =  0.05360
Epoch 2401: loss =  0.05361
Epoch 2501: loss =  0.05357
Epoch 2601: loss =  0.05355
Epoch 2701: loss =  0.05356
Epoch 2801: loss =  0.05352
Epoch 2901: loss =  0.05351
Epoch 3001: loss =  0.05351
Epoch 3101: loss =  0.05353
Epoch 3201: loss =  0.05354
Epoch 3301: loss =  0.05350
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_hoptimus_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_hoptimus_{model_type}.pth',weights_only=True))

### UNI

In [17]:
#UNI
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.10495
Epoch 101: loss =  0.08602
Epoch 201: loss =  0.06574
Epoch 301: loss =  0.05906
Epoch 401: loss =  0.05754
Epoch 501: loss =  0.05630
Epoch 601: loss =  0.05571
Epoch 701: loss =  0.05540
Epoch 801: loss =  0.05522
Epoch 901: loss =  0.05495
Epoch 1001: loss =  0.05472
Epoch 1101: loss =  0.05457
Epoch 1201: loss =  0.05445
Epoch 1301: loss =  0.05419
Epoch 1401: loss =  0.05403
Epoch 1501: loss =  0.05387
Epoch 1601: loss =  0.05370
Epoch 1701: loss =  0.05355
Epoch 1801: loss =  0.05352
Epoch 1901: loss =  0.05340
Epoch 2001: loss =  0.05331
Epoch 2101: loss =  0.05325
Epoch 2201: loss =  0.05320
Epoch 2301: loss =  0.05314
Epoch 2401: loss =  0.05308
Epoch 2501: loss =  0.05319
Epoch 2601: loss =  0.05300
Epoch 2701: loss =  0.05294
Epoch 2801: loss =  0.05291
Epoch 2901: loss =  0.05287
Epoch 3001: loss =  0.05285
Epoch 3101: loss =  0.05284
Epoch 3201: loss =  0.05278
Epoch 3301: loss =  0.05275
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [18]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_UNI_{model_type}.pth',weights_only=True))

### virchow

In [ ]:
#virchow
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.10496
Epoch 101: loss =  0.08793
Epoch 201: loss =  0.07042
Epoch 301: loss =  0.06138
Epoch 401: loss =  0.05884
Epoch 501: loss =  0.05720
Epoch 601: loss =  0.05622
Epoch 701: loss =  0.05563
Epoch 801: loss =  0.05521
Epoch 901: loss =  0.05497
Epoch 1001: loss =  0.05473
Epoch 1101: loss =  0.05447
Epoch 1201: loss =  0.05452
Epoch 1301: loss =  0.05416
Epoch 1401: loss =  0.05408
Epoch 1501: loss =  0.05398
Epoch 1601: loss =  0.05383
Epoch 1701: loss =  0.05376
Epoch 1801: loss =  0.05371
Epoch 1901: loss =  0.05366
Epoch 2001: loss =  0.05361
Epoch 2101: loss =  0.05365
Epoch 2201: loss =  0.05367
Epoch 2301: loss =  0.05364
Epoch 2401: loss =  0.05350
Epoch 2501: loss =  0.05349
Epoch 2601: loss =  0.05348
Epoch 2701: loss =  0.05346
Epoch 2801: loss =  0.05340
Epoch 2901: loss =  0.05349
Epoch 3001: loss =  0.05341
Epoch 3101: loss =  0.05343
Epoch 3201: loss =  0.05329
Epoch 3301: loss =  0.05327
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_virchow_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_virchow_{model_type}.pth',weights_only=True))

## Model Type 5

In [19]:
model_type = 5

HA.set_random_seed(42)
model = HA.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=32, model_type=model_type, n_scales=2)
model = model.to(device)


### NOISE

In [ ]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.10496
Epoch 101: loss =  0.08710
Epoch 201: loss =  0.06700
Epoch 301: loss =  0.06188
Epoch 401: loss =  0.06043
Epoch 501: loss =  0.05929
Epoch 601: loss =  0.05884
Epoch 701: loss =  0.05865
Epoch 801: loss =  0.05856
Epoch 901: loss =  0.05829
Epoch 1001: loss =  0.05795
Epoch 1101: loss =  0.05784
Epoch 1201: loss =  0.05780
Epoch 1301: loss =  0.05775
Epoch 1401: loss =  0.05773
Epoch 1501: loss =  0.05771
Epoch 1601: loss =  0.05769
Epoch 1701: loss =  0.05768
Epoch 1801: loss =  0.05767
Epoch 1901: loss =  0.05765
Epoch 2001: loss =  0.05764
Epoch 2101: loss =  0.05764
Epoch 2201: loss =  0.05762
Epoch 2301: loss =  0.05762
Epoch 2401: loss =  0.05760
Epoch 2501: loss =  0.05760
Epoch 2601: loss =  0.05759
Epoch 2701: loss =  0.05758
Epoch 2801: loss =  0.05757
Epoch 2901: loss =  0.05755
Epoch 3001: loss =  0.05755
Epoch 3101: loss =  0.05754
Epoch 3201: loss =  0.05754
Epoch 3301: loss =  0.05753
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_NOISE_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_NOISE_{model_type}.pth',weights_only=True))

### h_Optimus

In [ ]:
#H_OPTIMUS
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.10496
Epoch 101: loss =  0.08683
Epoch 201: loss =  0.06649
Epoch 301: loss =  0.05997
Epoch 401: loss =  0.05844
Epoch 501: loss =  0.05713
Epoch 601: loss =  0.05592
Epoch 701: loss =  0.05531
Epoch 801: loss =  0.05495
Epoch 901: loss =  0.05461
Epoch 1001: loss =  0.05450
Epoch 1101: loss =  0.05424
Epoch 1201: loss =  0.05417
Epoch 1301: loss =  0.05404
Epoch 1401: loss =  0.05398
Epoch 1501: loss =  0.05394
Epoch 1601: loss =  0.05386
Epoch 1701: loss =  0.05381
Epoch 1801: loss =  0.05381
Epoch 1901: loss =  0.05375
Epoch 2001: loss =  0.05367
Epoch 2101: loss =  0.05363
Epoch 2201: loss =  0.05360
Epoch 2301: loss =  0.05357
Epoch 2401: loss =  0.05356
Epoch 2501: loss =  0.05353
Epoch 2601: loss =  0.05352
Epoch 2701: loss =  0.05350
Epoch 2801: loss =  0.05353
Epoch 2901: loss =  0.05349
Epoch 3001: loss =  0.05349
Epoch 3101: loss =  0.05343
Epoch 3201: loss =  0.05339
Epoch 3301: loss =  0.05337
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_hoptimus_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_Hoptimus_{model_type}.pth',weights_only=True))

### UNI

In [20]:
#UNI
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.10496
Epoch 101: loss =  0.08680
Epoch 201: loss =  0.06641
Epoch 301: loss =  0.05994
Epoch 401: loss =  0.05837
Epoch 501: loss =  0.05699
Epoch 601: loss =  0.05598
Epoch 701: loss =  0.05523
Epoch 801: loss =  0.05472
Epoch 901: loss =  0.05436
Epoch 1001: loss =  0.05415
Epoch 1101: loss =  0.05400
Epoch 1201: loss =  0.05391
Epoch 1301: loss =  0.05388
Epoch 1401: loss =  0.05378
Epoch 1501: loss =  0.05373
Epoch 1601: loss =  0.05369
Epoch 1701: loss =  0.05362
Epoch 1801: loss =  0.05384
Epoch 1901: loss =  0.05354
Epoch 2001: loss =  0.05348
Epoch 2101: loss =  0.05345
Epoch 2201: loss =  0.05345
Epoch 2301: loss =  0.05338
Epoch 2401: loss =  0.05341
Epoch 2501: loss =  0.05335
Epoch 2601: loss =  0.05333
Epoch 2701: loss =  0.05330
Epoch 2801: loss =  0.05326
Epoch 2901: loss =  0.05323
Epoch 3001: loss =  0.05321
Epoch 3101: loss =  0.05322
Epoch 3201: loss =  0.05370
Epoch 3301: loss =  0.05317
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [21]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_UNI_{model_type}.pth',weights_only=True))

### virchow

In [ ]:
#virchow
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.10496
Epoch 101: loss =  0.08686
Epoch 201: loss =  0.06655
Epoch 301: loss =  0.06002
Epoch 401: loss =  0.05836
Epoch 501: loss =  0.05726
Epoch 601: loss =  0.05603
Epoch 701: loss =  0.05556
Epoch 801: loss =  0.05515
Epoch 901: loss =  0.05493
Epoch 1001: loss =  0.05481
Epoch 1101: loss =  0.05466
Epoch 1201: loss =  0.05454
Epoch 1301: loss =  0.05460
Epoch 1401: loss =  0.05439
Epoch 1501: loss =  0.05433
Epoch 1601: loss =  0.05420
Epoch 1701: loss =  0.05409
Epoch 1801: loss =  0.05401
Epoch 1901: loss =  0.05408
Epoch 2001: loss =  0.05387
Epoch 2101: loss =  0.05439
Epoch 2201: loss =  0.05371
Epoch 2301: loss =  0.05357
Epoch 2401: loss =  0.05354
Epoch 2501: loss =  0.05342
Epoch 2601: loss =  0.05340
Epoch 2701: loss =  0.05335
Epoch 2801: loss =  0.05330
Epoch 2901: loss =  0.05322
Epoch 3001: loss =  0.05320
Epoch 3101: loss =  0.05315
Epoch 3201: loss =  0.05311
Epoch 3301: loss =  0.05308
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(al

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_32_virchow_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_32_virchow_{model_type}.pth',weights_only=True))